# Create Agent
Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.

In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm = ChatOllama(model="deepseek-r1:8b")
agent = create_agent(model=llm)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the capital of France?"}]}
)
for message in response["messages"]:
    print(message.content)

What is the capital of France?
The capital of France is **Paris**.


# Create Agent with System Message
Agents with system messages allow you to provide instructions or context to the agent, guiding its behavior and responses. This can help the agent understand the task better and generate more relevant outputs.

In [2]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

llm = ChatOllama(model="deepseek-r1:8b")
agent = create_agent(model=llm, system_prompt="You are a helpful assistant.")
response = agent.invoke({"messages": [{"role": "user", "content": "What is 1/0?"}]})
for message in response["messages"]:
    print(message.content)

What is 1/0?
1/0 is **undefined**.

Here's why:

*   Division means finding how many times one number (the divisor) fits into another number (the dividend). For example, 10/5 = 2 because 5 fits into 10 two times.
*   When you try to divide by zero, you're asking: "What number, when multiplied by 0, gives you 1?"
*   However, any number multiplied by 0 is 0. There is no number that you can multiply by 0 to get 1. Therefore, the answer does not exist.

This is true for any division by zero, not just 1/0. For example, 5/0 is also undefined.


## Create Agent with Dynamic System Message

In [3]:
from langchain.agents import create_agent, AgentState
from langchain_ollama import ChatOllama
from langchain.agents.middleware import dynamic_prompt

llm = ChatOllama(model="gemma4:e4b")


@dynamic_prompt
def user_role_prompt(state: AgentState) -> str:
    """Generate system prompt based on user role."""
    message_length = len(state.messages[-1].content)
    base_prompt = "You are a helpful assistant."

    if message_length > 10:
        return f"{base_prompt} Provide answers in a concise manner.Give short explanation not more than 1000 characters."
    else:
        return (
            f"{base_prompt} Provide answers in a concise manner.No explanation needed."
        )


print("#" * 50, "Testing with long message", "#" * 50)
agent = create_agent(model=llm, middleware=[user_role_prompt])
response = agent.invoke(
    input={"messages": [{"role": "user", "content": "What is 1/0?"}]}
)
for message in response["messages"]:
    print(message.content)

print("#" * 50, "Testing with short message", "#" * 50)
response = agent.invoke(input={"messages": [{"role": "user", "content": "1/0 = ?"}]})
for message in response["messages"]:
    print(message.content)

################################################## Testing with long message ##################################################
What is 1/0?
$1/0$ is **undefined** in standard mathematics.

Division is the inverse of multiplication. Asking what $1/0$ is means asking, "What number, when multiplied by 0, equals 1?" There is no such number, because any number multiplied by zero is always zero, not one.

In advanced mathematics (like calculus), approaching zero can lead to concepts of "infinity" ($\pm\infty$), but the simple arithmetic expression $1/0$ remains undefined.
################################################## Testing with short message ##################################################
1/0 = ?
Undefined
